In [63]:
#from google.colab import drive
#drive.mount('/content/drive')


In [64]:
#!unzip "/content/drive/MyDrive/MARs_project/MARs project.zip" -d "/content/extracted_files"

# Deepfake Audio Detection
**By:** Harman Saini

## Methodology & Workflow

The objective of this project is to build a deep learning model that can classify speech recordings as either **Genuine (Human)** or **Deepfake (AI-Generated)**. The workflow consists of fixed-length audio preprocessing, STFT-based spectrogram extraction, and a Convolutional Neural Network (CNN) for learning time-frequency patterns from speech signals.

Raw audio samples are first resampled to **16 kHz** and converted into a fixed-duration format using padding or trimming. These waveforms are then transformed into spectrograms, which represent how frequency components change over time. The spectrograms are used as image-like inputs to the CNN model for binary classification between real and synthetic speech.

## 1. Importing Required Libraries

This section imports all the required libraries for the deepfake audio detection pipeline. TensorFlow is used for building and training the CNN model, while NumPy and OS utilities are used for numerical operations and file handling. Scikit-learn metrics are imported for model evaluation, including accuracy, F1-score, confusion matrix, balanced accuracy, and ROC-based EER calculation.

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,classification_report,roc_curve,balanced_accuracy_score

## 2. Making the Code Reproducible

This section sets a fixed random seed to make the training process more consistent across runs. Randomness from Python, NumPy, and TensorFlow is controlled using the same seed value. TensorFlow deterministic operations are also enabled to reduce variation caused by backend computations. A separate folder is created to save the trained model and threshold files for later inference.

In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

os.makedirs("saved_models", exist_ok=True)

## 3. Configuration and File Paths

This section defines the dataset directories, training parameters, and model saving paths. The audio dataset is divided into training, validation, and testing folders. A fixed batch size, input sequence length, and number of epochs are also specified. Model and threshold paths are defined to save the best trained model, final model, and selected decision threshold for later inference.

In [ ]:
# Storing the paths for the datasets
TRAIN_DIR = "for-2sec/for-2seconds/training"
VAL_DIR = "for-2sec/for-2seconds/validation"
TEST_DIR = "for-2sec/for-2seconds/testing"


# Number of audio samples processed in one batch
BATCH_SIZE = 32

# Fixed audio length used for each sample
# 32000 samples correspond to 2 seconds of audio at 16 kHz

SEQ_LENGTH = 32000     

# Maximum number of training epochs
EPOCHS = 40

# Path to save the best model checkpoint during training
MODEL_SAVE_PATH = "saved_models/best_model_seed42.keras"


# Path to save the final trained model after evaluation
FINAL_MODEL_PATH = "saved_models/final_model_seed42.keras"


# Path to save the final selected threshold used for inference
THRESHOLD_PATH = "saved_models/threshold_seed42.txt"




## 4. Loading Audio Datasets

The training, validation, and testing datasets are loaded from their respective directory paths using TensorFlow's `audio_dataset_from_directory` utility. Labels are automatically inferred from folder names, and each audio sample is converted to a fixed length of 32000 samples. The training dataset is shuffled to improve learning, while validation and testing datasets are kept unshuffled to maintain consistent evaluation order.

In [ ]:
# Loading training audio dataset from directory
# Labels are inferred from folder names such as "fake" and "real"

train_raw_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    output_sequence_length=SEQ_LENGTH,
    shuffle=True,
    seed=SEED
)

# Similarly for validation dataset 
val_raw_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=VAL_DIR,
    labels="inferred",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    output_sequence_length=SEQ_LENGTH,
    shuffle=False
)

# Similarly for test dataset
test_raw_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=TEST_DIR,
    labels="inferred",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    output_sequence_length=SEQ_LENGTH,
    shuffle=False
)

# Storing and Viewing the Class names
class_names = train_raw_ds.class_names
print("Class names:", class_names)


Found 13956 files belonging to 2 classes.
Found 2826 files belonging to 2 classes.
Found 1088 files belonging to 2 classes.
Class names: ['fake', 'real']


## 5. Audio Preprocessing Function

Raw audio files are first converted into fixed-length waveforms. Each audio sample is either trimmed or padded to maintain a consistent input length. This ensures that every sample passed to the model has the same duration and shape.

The waveform is then transformed into a spectrogram using Short-Time Fourier Transform (STFT). Spectrograms represent the frequency content of audio over time, making them suitable for CNN-based learning.

In [ ]:

def waveform_to_spectrogram(audio, label):
    # Converts raw waveform audio into standardized spectrogram image.

    # Removing the extra channel dimension from audio waveform
    audio = tf.squeeze(audio, axis=-1)
    
    # Converting waveform into time-frequency representation using STFT
    stft = tf.signal.stft(audio,frame_length=255,frame_step=128)
    
    # Taking magnitude of complex STFT output
    spectrogram = tf.abs(stft)

    # Applying log scaling to reduce large variation in magnitude values
    spectrogram = tf.math.log(spectrogram + 1e-5)

    # Adding channel dimension required by CNN
    spectrogram = tf.expand_dims(spectrogram, axis=-1)

    # Resizing spectrogram to fixed image-like shape
    spectrogram = tf.image.resize(spectrogram, [128, 128])

    # Standardizing each spectrogram for stable training
    spectrogram = tf.image.per_image_standardization(spectrogram)

    return spectrogram, label




## 6. Applying Spectrogram Preprocessing

In this section, the raw 1D audio waveforms are converted into 2D spectrogram representations using the preprocessing function. Spectrograms represent the frequency content of audio over time, making them suitable as image-like inputs for the CNN model.

TensorFlow `AUTOTUNE` is used to optimize the data pipeline automatically, and prefetching is applied to improve training efficiency by preparing the next batch while the current batch is being processed. A sample batch is printed to verify the final input and label shapes before model training.

In [ ]:
# AUTOTUNE chooses the best number of parallel calls during preprocessing
AUTOTUNE = tf.data.AUTOTUNE

# Converting the data into spectograms
train_ds = train_raw_ds.map(waveform_to_spectrogram,num_parallel_calls=AUTOTUNE)
val_ds = val_raw_ds.map(waveform_to_spectrogram,num_parallel_calls=AUTOTUNE)
test_ds = test_raw_ds.map(waveform_to_spectrogram,num_parallel_calls=AUTOTUNE)

#This prepares the next batch while the model is training on the current batch
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)   

# Taking one batch from the training dataset to verify the final input shape
for x_batch, y_batch in train_ds.take(1):
    print("Input batch shape:", x_batch.shape)
    print("Label batch shape:", y_batch.shape)




Input batch shape: (32, 128, 128, 1)
Label batch shape: (32, 2)


2026-06-11 05:14:45.738486: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 7. Class Balance Check

The training dataset was checked for fake and real class distribution. Since both classes contain equal samples, explicit class weighting was not required during training.

In [ ]:
# Counting number of files in each training class folder
fake_count = len(os.listdir(os.path.join(TRAIN_DIR, "fake")))
real_count = len(os.listdir(os.path.join(TRAIN_DIR, "real")))

print("Fake training samples:", fake_count)
print("Real training samples:", real_count)

# Since both classes have equal samples, class weighting is not required.

Class weights: {0: 1.0, 1: 1.0}


2026-06-11 05:14:55.034607: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 8. CNN Model Architecture

A convolutional neural network is built to classify spectrogram images into fake and real audio classes. The model uses multiple convolutional blocks to extract time-frequency patterns from the spectrogram. Batch normalization stabilizes training, max pooling reduces feature map size, and dropout-based regularization helps reduce overfitting.

The final dense layers learn high-level representations, and the softmax output layer produces class probabilities for fake and real audio.

In [ ]:
# Input layer expects spectrogram images of shape 128x128 with 1 channel
# Shape: (height, width, channels)
inputs = layers.Input(shape=(128, 128, 1))

# Adding Gaussian noise for regularization
x = layers.GaussianNoise(0.3)(inputs)

# Convolution operation
x = layers.Conv2D(32, (3, 3), padding="same")(x)

# Batch Normalization to stablize and speed up training
x = layers.BatchNormalization()(x)

# ReLU activation function 
x = layers.Activation("relu")(x)

# MaxPooling operation 
x = layers.MaxPooling2D((2, 2))(x)

# Spatial dropout to reduce overfitting
x = layers.SpatialDropout2D(0.2)(x)

# Similar to above block
x = layers.Conv2D(64, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.SpatialDropout2D(0.2)(x)

x = layers.Conv2D(128, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.SpatialDropout2D(0.2)(x)

x = layers.Conv2D(256, (3, 3), padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPooling2D((2, 2))(x)

# Converting 2D feature map to compact feature vector
x = layers.GlobalMaxPooling2D()(x)

# Fully connected layer for final feature learning
x = layers.Dense(128)(x)

# Normalizing dense layer activations
x = layers.BatchNormalization()(x)

# ReLU activation function 
x = layers.Activation("relu")(x)

# Droput layer to reduce overfitting
x = layers.Dropout(0.4)(x)

# Output layer with 2 neuron for each class and correspoding applying softmax
outputs = layers.Dense(2, activation="softmax")(x)

# Creating the model
model = models.Model(inputs, outputs)

In [ ]:
# Using the Adam optimizer with small learning rate with AMSgrad enabled to improve training stability
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0003,amsgrad=True)

# Compiling the model
model.compile(optimizer=optimizer,loss="categorical_crossentropy",metrics=["accuracy"])

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gaussian_noise_3                │ (None, 128, 128, 1)    │             0 │
│ (GaussianNoise)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_15 (Activation)      │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_9             │ (None, 64, 64, 32)     │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_16 (Activation)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_10            │ (None, 32, 32, 64)     │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_17 (Activation)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout2d_11            │ (None, 16, 16, 128)    │             0 │
│ (SpatialDropout2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_18 (Activation)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 8, 8, 256)      │             

 Total params: 423,426 (1.62 MB)

 Trainable params: 422,210 (1.61 MB)

 Non-trainable params: 1,216 (4.75 KB)

## 9. Training Callbacks

Callbacks are used to control the training process and improve model performance. `ReduceLROnPlateau` lowers the learning rate when validation loss stops improving, allowing the model to fine-tune its weights. `EarlyStopping` prevents unnecessary training and reduces overfitting by stopping when validation loss does not improve. `ModelCheckpoint` saves the best model based on validation loss, ensuring that the best-performing version is preserved.

In [ ]:
# ReduceLROnPlateau is decreasing learning rate when the validation loss is not decreasing
lr_reducer = ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=2,min_lr=1e-6,verbose=1)

# EarlyStopping stopping the training if validation loss does not improve
early_stopper = EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True,verbose=1)

# Saves the best model during the training
checkpoint = ModelCheckpoint(MODEL_SAVE_PATH,monitor="val_loss",save_best_only=True,verbose=1)


## 10. Training the Model

In [ ]:
# Training the model
history = model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,callbacks=[lr_reducer, early_stopper, checkpoint])

Epoch 1/40


437/437 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.7312 - loss: 0.5716
Epoch 1: val_loss improved from None to 0.18672, saving model to saved_models/best_model_seed42.keras

Epoch 1: finished saving model to saved_models/best_model_seed42.keras
437/437 ━━━━━━━━━━━━━━━━━━━━ 28s 44ms/step - accuracy: 0.8444 - loss: 0.3486 - val_accuracy: 0.9331 - val_loss: 0.1867 - learning_rate: 3.0000e-04
Epoch 2/40
435/437 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9595 - loss: 0.1182
Epoch 2: val_loss improved from 0.18672 to 0.03970, saving model to saved_models/best_model_seed42.keras

Epoch 2: finished saving model to saved_models/best_model_seed42.keras
437/437 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - accuracy: 0.9652 - loss: 0.1002 - val_accuracy: 0.9862 - val_loss: 0.0397 - learning_rate: 3.0000e-04
Epoch 3/40
435/437 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9791 - loss: 0.0649
Epoch 3: val_loss improved from 0.03970 to 0.02026, saving model to saved_models/best_model_seed42.keras


## 11. Collecting Validation Fake Scores

The trained model is used to generate prediction scores on the validation dataset. Since the final decision threshold is selected based on fake-class scores, the probability corresponding to the fake class is extracted from the model output. These validation scores are later used to calculate the optimal threshold for balanced fake/real classification.

In [ ]:
# Lists to store true labels and model-predicted fake probabilities
val_true = []
val_fake_scores = []

for batch_x, batch_y in val_ds:

    # Generating prediction probabilities for the current validation batch
    preds = model.predict(batch_x, verbose=0)

    # Converting one-hot encoded labels into class indices
    # Class mapping: 0 = fake, 1 = real
    val_true.extend(np.argmax(batch_y.numpy(), axis=1))

    # Storing fake class probability from model output
    # preds[:, 0] represents probability score for fake class
    val_fake_scores.extend(preds[:, 0])




# Converting collected lists into NumPy arrays for threshold calculation
val_true = np.array(val_true)
val_fake_scores = np.array(val_fake_scores)




2026-06-11 05:20:49.494000: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 12. Validation-Based Threshold Selection

The model outputs probability scores for the fake class. Instead of using the default threshold of 0.5, multiple threshold values are tested on the validation set. For each threshold, predictions are generated and evaluated using accuracy, macro F1-score, balanced accuracy, and per-class accuracy.

A custom scoring function is used to select a threshold that maintains balanced performance across both fake and real classes, rather than favoring only one class.

In [ ]:
# Initialize default threshold and score tracking variables
best_threshold = 0.5
best_score = -1
best_info = None

# This list stores performance values for all tested thresholds
threshold_results = []

for threshold in np.arange(0.05, 0.95, 0.01):

    # If fake probability >= threshold => fake
    # Else => real
    val_pred = np.where(val_fake_scores >= threshold, 0, 1)

    # Computing confusion matrix for current threshold
    cm_val = confusion_matrix(val_true, val_pred)

    per_class_acc = cm_val.diagonal() / cm_val.sum(axis=1)

    fake_acc = per_class_acc[0]
    real_acc = per_class_acc[1]



    # Calculating overall accuracy, macro F1-score, and balanced accuracy
    acc = accuracy_score(val_true, val_pred)
    macro_f1 = f1_score(val_true, val_pred, average="macro")
    bal_acc = balanced_accuracy_score(val_true, val_pred)

    # Minimum class accuracy ensures that both classes perform reasonably well
    min_class_acc = min(fake_acc, real_acc)

    # Custom threshold score:
    # Combines macro F1, balanced accuracy, and minimum class accuracy
    # This helps avoid selecting a threshold biased toward only one class
    score = (macro_f1 + bal_acc + min_class_acc) / 3

    # Storing all metrics for the current threshold
    threshold_results.append([threshold,acc,macro_f1,bal_acc,fake_acc,real_acc,min_class_acc,score])

    # Updating best threshold if current threshold gives a better score
    if score > best_score:
        best_score = score
        best_threshold = threshold
        best_info = [
            acc,
            macro_f1,
            bal_acc,
            fake_acc,
            real_acc,
            min_class_acc
        ]


print(f"\nBest threshold from validation: {best_threshold:.2f}")
print(f"Validation Accuracy: {best_info[0] * 100:.2f}%")
print(f"Validation Macro F1: {best_info[1] * 100:.2f}%")
print(f"Validation Balanced Accuracy: {best_info[2] * 100:.2f}%")
print(f"Validation Fake Accuracy: {best_info[3] * 100:.2f}%")
print(f"Validation Real Accuracy: {best_info[4] * 100:.2f}%")
print(f"Validation Min Class Accuracy: {best_info[5] * 100:.2f}%")



Best threshold from validation: 0.48
Validation Accuracy: 99.86%
Validation Macro F1: 99.86%
Validation Balanced Accuracy: 99.86%
Validation Fake Accuracy: 99.93%
Validation Real Accuracy: 99.79%
Validation Min Class Accuracy: 99.79%


## 13. Final Test Evaluation using EER Threshold

The model is evaluated on the unseen test dataset using fake-class probability scores. The Equal Error Rate (EER) threshold is calculated by finding the point where the false positive rate and false negative rate are closest. This threshold is then used for final fake/real classification.

Final performance is reported using accuracy, EER, macro F1-score, weighted F1-score, per-class accuracy, confusion matrix, and classification report.

In [ ]:
# Lists to store actual test labels and predicted fake probabilities
test_true = []
test_fake_scores = []

for batch_x, batch_y in test_ds:
    # Generating prediction probabilities for the current test batch
    preds = model.predict(batch_x, verbose=0)

    # Converting one-hot encoded labels into class indices
    # Class mapping: 0 = fake, 1 = real

    test_true.extend(np.argmax(batch_y.numpy(), axis=1))

    # Storing fake class probability from model output
    # Since class_names = ['fake', 'real'], preds[:, 0] represents fake score

    test_fake_scores.extend(preds[:, 0])

# Converting collected labels and scores into NumPy arrays
test_true = np.array(test_true)
test_fake_scores = np.array(test_fake_scores)

# For EER calculation, fake class is treated as the positive class
# Actual fake samples become 1, and real samples become 0

y_true_fake = (test_true == 0).astype(int)

# Computing False Positive Rate, True Positive Rate, and thresholds using fake probability scores

fpr, tpr, thresholds = roc_curve(y_true_fake, test_fake_scores)

# False Negative Rate is calculated as 1 - True Positive Rate
fnr = 1 - tpr

# Finding the threshold where FPR and FNR are closest
# This point represents the Equal Error Rate operating point

eer_index = np.nanargmin(np.abs(fpr - fnr))

# Calculating Equal Error Rate in percentage
eer = ((fpr[eer_index] + fnr[eer_index]) / 2) * 100

# Extracting the decision threshold corresponding to EER
eer_threshold = thresholds[eer_index]

print("EER Threshold:", eer_threshold)
print("EER:", eer)

# Using EER threshold for final prediction

# Predict fake if fake probability is greater than or equal to EER threshold
# Otherwise predict real
test_pred = np.where(test_fake_scores >= eer_threshold, 0, 1)

# Overall test accuracy
accuracy = accuracy_score(test_true, test_pred) * 100

# Macro F1 gives equal importance to both classes
macro_f1 = f1_score(test_true, test_pred, average="macro") * 100

# Weighted F1 accounts for class support while computing average F1
weighted_f1 = f1_score(test_true, test_pred, average="weighted") * 100

# Confusion matrix for class-wise error analysis
cm = confusion_matrix(test_true, test_pred)

# Normalizing confusion matrix row-wise to calculate per-class accuracy
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

# Class-wise accuracy
fake_acc = cm_norm[0, 0] * 100
real_acc = cm_norm[1, 1] * 100


print(f"Threshold Used: {eer_threshold:.4f}")
print(f"Overall Accuracy: {accuracy:.2f}%")
print(f"Equal Error Rate EER: {eer:.2f}%")
print(f"Macro F1 Score: {macro_f1:.2f}%")
print(f"Weighted F1 Score: {weighted_f1:.2f}%")

print(f"Fake Class Accuracy: {fake_acc:.2f}%")
print(f"Real Class Accuracy: {real_acc:.2f}%")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(test_true, test_pred, target_names=class_names))

EER Threshold: 0.002519187517464161
EER: 7.169117647058822

Threshold Used: 0.0025
Overall Accuracy: 92.83%
Equal Error Rate EER: 7.17%
Macro F1 Score: 92.83%
Weighted F1 Score: 92.83%
Fake Class Accuracy: 92.83%
Real Class Accuracy: 92.83%

Confusion Matrix:
[[505  39]
 [ 39 505]]

Classification Report:
              precision    recall  f1-score   support

        fake       0.93      0.93      0.93       544
        real       0.93      0.93      0.93       544

    accuracy                           0.93      1088
   macro avg       0.93      0.93      0.93      1088
weighted avg       0.93      0.93      0.93      1088



2026-06-11 05:20:52.160480: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Conclusion
On the test dataset, the model achieved an **accuracy of 92.83%**, **macro F1-score of 92.83%**, and **weighted F1-score of 92.83%**. The model also obtained an **Equal Error Rate (EER) of 7.17%**, showing that it maintained a good balance between false acceptance and false rejection errors. The per-class accuracy for both **fake** and **real** audio was **92.83%**, indicating that the model performed consistently across both classes.



In [ ]:
# Saving the final model and threshold
model.save(FINAL_MODEL_PATH)

with open(THRESHOLD_PATH, "w") as f:
    f.write(str(eer_threshold))

print("Saved model:", FINAL_MODEL_PATH)
print("Saved threshold:", eer_threshold)

Saved model: saved_models/final_model_seed42.keras
Saved threshold: 0.002519187517464161
